# Ilastik classification

This notebook covers Ilastik object classification and pixel classification for the MACSima spatial proteomics dataset. It continues from `6.proteomics_macsima.ipynb`, assuming the segmented `SpatialData` object is already available.

## 1. Ilastik object classification

@ Julien -> we can replace with napari_harpy object classification. I can show you how it works, and then we can see if napari_harpy would still miss something. 
We can keep the Ilastik documentation as a reference, but then we move it to a seperate notebook, together with the pixel classification.

In [ ]:
from ilastik.sdata_to_ilastik import (
    export_h5,
    add_ilastik_to_sdata,
    assign_ilastik_cell_types,
    combine_ilastik_predictions,
)

### 1.1 Export images for Ilastik

In [ ]:
# Paths to export h5 files
img_layer_to_export = "REAscreen_IO_CRC_image"
path_to_raw_h5 = os.path.join(output_path, f"raw_images.h5")

segmentation_mask_to_export = "labels_cells_instanseg_artifacts_filtered"
path_to_segmentation_h5 = os.path.join(output_path, "segmentation_mask.h5")

# Export raw image channels
export_h5(
    sdata=sdata_crop,
    img_layer=img_layer_to_export,
    channels=None,  # Will export all channels
    output=path_to_raw_h5,
    crd=None,  # Images will not be cropped
)

# Export segmentation mask
export_h5(
    sdata=sdata_crop,
    labels_layer=segmentation_mask_to_export,
    output=path_to_segmentation_h5,
    crd=None,
)

# NOTE: To avoid crashing Ilastik, the images should not be too large.

### 1.2 Training Ilastik object classifiers

There are a variety of diffent object classifiers that can be created in ilastik.

Some options:
- You can train on a single image or on a combination of images (but more images will increase complexity and computation time)
- You can train on any set of features that is most appropriate (but more features will increase complexity and computation time)
- You can have as many classes as is needed (but more classes will increase complexity and computation time)
- As a rule of thumb, you can classify for anything as long as you can visually recognize it yourself on an image (or a set of images)

For example:
- You can use the nucleus channel to classify cells in good segmentations and bad segmentations.
- You can train for each channel a classifier that classifies cells in positive and negative for that marker.
- You can train on multiple channels simultaneously to classify cells in different cell types.

For each object classifier, do the following:

Open Ilastik (v.1.4.0) and create an new project: Object Classification [Inputs: Raw Data, Segmentation]

Note that this documentation was written for version v.1.4.0, but everything seems to work with the latest version (v.1.4.1) as well.

**1. Input Data** </br>
To load in a separate channel, you need to do the following:

- Under the tab `Raw Data` from `1. Input Data`, you would click `Add...` and select `Add separate Image(s)...`.
- Select the .h5 file containing the raw images and you will need to specify which channel you want to work on from the drop-down menu and click `OK` (this specifies the internal path in the h5 file).

To load in multiple channels, you need to specify a correct pattern that also includes the internal path (i.e. in the h5 file) to the correct images.

For example, to combine channel_1 and channel_2, you would do the following steps:
- Under the tab 'Raw Data' from '1. Input Data', you would click 'Add...' and select 'Add a single 3D/4D Volume from Sequence...'.
- Under `Specify Pattern`, you enter a patterns that specifies the images of interest. For example: `/path/to/images/raw_images.h5/channel_1; /path/to/images/raw_images.h5/channel_2` and click `Apply`. It is important that the path is specified correctly, multiple paths should be separated by a semicolon and the internal path in the h5 file should be specified as well in the path.
- Select `Stack Across: C` before clicking `OK`.

You will also need to add the segmentation mask. Under the tab `Segmentation Image`, you click `Add...` and select `Add separate Image(s)...`. You then select the appropriate segmentation mask file (e.g. `segmentation_mask.h5`).

After adding the Raw Data and the Segmentation Image to the ilastik project, it is useful to right-click on them, go to `Edit properties...` and make sure `Storage:` is set to `Copy into project file`. This makes sure, you can move around the ilastik project file (on your computer or even to other computers) without losing the link to the files the project was trained on (and risk losing your training). It is also useful to set `Nickname:` to something informative such as the name of the tissue/sample/replicate/etc (to keep track of which image is which). By default, this will be set to the filename.

<b>Important:</b> Note that the images for training in the Ilastik project can't be too large or Ilastik will crash. Around 10000 x 10000 pixels seems to work fine, but smaller (for training at least) is preferred. 

**2. Object Feature Selection** </br>
To select features for training, it would be recommended to not select all of them, but be mindful of which you want to train on. Although ilastik itself describes computing many features at once as computationally cheap, it can still really add up to calculate all features if there are a lot of cells in each image. Additionally, by, for example, only working on intensity-related features, it becomes more explainable what the model was trained on how it can be interpreted.

For most cases, we would recommend to follow these steps:
- Under the tab `2. Object Feature Selection`, click `Select Features` and click all boxes under `Intensity Distribution`. You can add other features that you know are relevant, if needed (such as `Size in pixels` or `Diameter`). In case you want create a classifier to distinguish good segmentations from bad segmentation, it would make sense to select all features.
- After clicking `OK`, wait until all features have been computed before moving on to the next step.

<b>Important:</b> Adding too many features (i.e. more than is necessary to get good classification) risks crashing the Ilastik project. This can also create problems for large images when running headless mode. Working on only the intensity features seems to work fine.

**3. Object Classification** </br>
Here, you can create multiple classes for your classifier and train them until you are satisfied with the results. In general, it is useful to initially add a good amount of labels for the different classes for different regions of the image (or even already over multiple images) to capture the variation that is in the data and click `Live Update` to see the prediction results. Subsequently, you can focus more on the mistakes that are being made and add labels to correct those. When labeling, we suggest to unclick `Live Update` to avoid waiting time. It can be useful to use the `Uncertainty` layer to see which objects are still not robustly trained for. When training, ilastik does not make it easy to change the visualization, but if you right-click on `Raw Input` and select `Adjust Thresholds`, you have some options to change the display. When training on individual channels for positive/negative, preferably use the following format to name the classes: channel_pos and channel_neg (e.g.: CD45_pos and CD45_neg).

Note that you can save your training labels by right-clicking `Labels` and subsequently `Export...`. Set the `File` to where you want to save the labels and under what name. This entire process needs to be done for every image separately, but it can be worth it to be able to recreate the ilastik project if something goes wrong or if we would want to preprocess the images for example. Of course, this only works with the exact same segmentation masks (so not after changing segmentation settings or working on a different image crop).

**4. Object Information Export** </br>
<b>Important</b>: You need to click `Configure Feature Table Export` and change `Format` to `CSV (.csv)`. You also need to set these settings when you don't want to export the results from the GUI, but will use headless mode in, for example, a Jupyter notebook. The headless mode requires this setting to be set in the GUI so if you don't do it, you will get an error later on. 

If you do want to export the results for the training data, you can specify under `Choose File` where and under what name you want to export the results. By default, this is set to `{dataset_dir}/{nickname}.csv`, with `{dataset_dir}` refering to the directory the raw_images.h5 file is in and `{nickname}` refering to the name specified in the `1. Input Data` tab. Preferably, remove `{nickname}` and put a unique name in its place for each ilastik classifier. For example: for a classifier to label cells as positive/negative for channel_1, put `{dataset_dir}/channel_1.csv`.

Note that, by default, the object predictions will be saved as images as well, while these files will not be used in the subsequent analysis steps (since we will get all useful data from the csv files). Unfortunately, there is no way to avoid saving these files.

**5. Batch processing** </br>
We will not use the batch processing tab since it is not possible to specify the internal paths in the h5 files.

**6. Blockwise Object classification** </br>
<b>Important</b>: Before closing the project, you need to set the Blocks to, for example, 4000 x 4000 and the Halo to 100 x 100 and save the project. This is necessary for running the headless mode since it gets these settings from the Ilastik project and you can't set the settings in headless mode itself.

In [ ]:
# Specify paths for Ilastik object classifiers
ilastik_obj_classifiers = {
    "CD8a": {  # Name of Ilastik object classifier. This will be used as an identifier for the classifier throughout the analysis
        "path": r"d:\example_data_MACSima\REAscreen_IO_CRC\output_harpy\CD8a.ilp",  # Path to Ilastik object classifier.
        "raw_images_internal_path_list": [
            "CD8a"
        ],  # Internal path in h5 file that was used during Ilastik training.
        "csv_path": output_path,  # Path to where you manually exported the CSV from the GUI, or where you want the headless mode to export the CSV to.
        "headless": False,  # Whether to use headless mode to creat the CSV
    },
    "CD3": {  # Name of Ilastik object classifier. This will be used as an identifier for the classifier throughout the analysis
        "path": r"d:\example_data_MACSima\REAscreen_IO_CRC\output_harpy\CD3.ilp",  # Path to Ilastik object classifier.
        "raw_images_internal_path_list": [
            "CD3"
        ],  # Internal path in h5 file that was used during Ilastik training.
        "csv_path": output_path,  # Path to where you manually exported the CSV from the GUI, or where you want the headless mode to export the CSV to.
        "headless": False,  # Whether to use headless mode to creat the CSV
    },
}

### 1.3 OPTIONAL: Creating CSV-files using Ilastik headless mode
This is a useful option if you created an object classifier and you want to use it with data that is not in the training set. Headless mode will then use the existing model and create a CSV-file for the new dataset.

Headless mode is also very useful if your image is too large to reliably work with the Ilastik GUI. You can then train the classifier on a crop of the data and use headless mode to get the results for the entire image.

In [ ]:
import subprocess

# Specify path to Ilastik installation
path_to_ilastik_exe = r"c:\Program Files\ilastik-1.4.1.post1\ilastik.exe"  # NOTE: Set this to the path to your Ilastik installation

# Run through all classifiers
for classifier_name, classifier_dict in ilastik_obj_classifiers.items():
    classifier_path = classifier_dict["path"]
    raw_images_internal_path_list = classifier_dict["raw_images_internal_path_list"]
    csv_path = classifier_dict["csv_path"]
    headless = classifier_dict["headless"]

    # Skip classifier if headless == False
    if not headless:
        print(f"skipping: {classifier_name}")
        continue

    # Expected output CSV filename (with '_table.csv' suffix)
    actual_csv_filename = os.path.join(
        csv_path, classifier_name + "_table.csv"
    )  # This is different from the '--table_filename' path because Ilastik always adds '_table' to the filename.

    # Check if output file already exists (possibly from exporting training data from the Ilastik GUI)
    if os.path.exists(actual_csv_filename):
        raise FileExistsError(f"{actual_csv_filename} already exists.")

    # Run ilastik headless as a subprocess
    cmd = [
        path_to_ilastik_exe,
        "--headless",
        "--project=" + classifier_path,
        "--raw_data="
        + ";".join(
            [f"{path_to_raw_h5}/{channel}" for channel in raw_images_internal_path_list]
        ),
        "--segmentation_image=" + path_to_segmentation_h5 + segmentation_mask_to_export,
        "--output_format=" + "png",
        "--export_source="
        + "Blockwise Object Predictions",  # Block size has to be set in the Ilastik project. We recommmend Blocks of 4000x4000 pixel and a Halo of 100x100 pixels.
        "--table_filename="
        + os.path.join(
            csv_path, classifier_name + ".csv"
        ),  # CSV has to be set as the export format in the Ilastik project.
    ]

    print(f"Running: {classifier_name}")
    result = subprocess.run(cmd, capture_output=True, text=True)

    ## OPTIONAL: Print the output and errors for debugging
    print("stdout:", result.stdout)
    print("stderr:", result.stderr)

    # Check if CSV file has been created
    if os.path.exists(actual_csv_filename):
        print(f"Output CSV file {actual_csv_filename} has been created successfully.")
    else:
        raise ValueError(f"Output CSV file {actual_csv_filename} was not created.")

# NOTE:
#  - When running this cell, make sure to close the Ilastik GUI since the headless mode will give an error if one of the ilastik projects is opened.
#  - Ilastik is very optimistic and will always tell you "The operation completed successfully.", even when there was an error. This is why we check whether the file was actually created.

### 1.4 Adding Ilastik data back to SpatialData

In [ ]:
# Add data from all ilastik csv files in folder to sdata
for classifier_name, classifier_dict in ilastik_obj_classifiers.items():
    # Get path to CSV
    csv_path = classifier_dict["csv_path"]
    actual_csv_filename = os.path.join(csv_path, classifier_name + "_table.csv")

    # Add ilastik results to sdata
    print("Running: ", classifier_name)
    add_ilastik_to_sdata(
        sdata=sdata_crop,
        input_path=actual_csv_filename,
        table_layer="table_intensities_leiden",
        labels_layer=segmentation_mask_to_export,
        centroid_column_x="centroid_x_cells",
        centroid_column_y="centroid_y_cells",
        suffix=classifier_name,
    )

# NOTE: This code will attempt to merge the data obtained from the ilastik classifiers to the sdata.tables[table_layer].obs based on the centroid coordinates of the sdata cells and the ilastik objects (i.e. for each cell in the sdata, the closest cell in the ilastik data will be considered a match).
# The suffix parameter is used to add a unique identifier to all columns associated with each ilastik dataset.


In [ ]:
# Inspect table layer obs
sdata_crop.tables["table_intensities_leiden"].obs

In [ ]:
# Plot user labels
from matplotlib.colors import ListedColormap

cmap = ListedColormap(["#0082C8", "#848484", "#FFE119"])

hp.pl.plot_shapes(
    sdata_crop,
    img_layer="REAscreen_IO_CRC_image",
    channel="DAPI (1)",
    shapes_layer="shapes_cells_instanseg_artifacts_filtered",
    table_layer="table_intensities_leiden",
    alpha=1,
    linewidth=0.6,
    cmap=cmap,
    column="ilastik_user_label_CD8a",
)
# There will be no user labels if you use headless mode

In [ ]:
# Plot predictions
cmap = ListedColormap(["#0082C8", "#FFE119"])

hp.pl.plot_shapes(
    sdata_crop,
    img_layer="REAscreen_IO_CRC_image",
    channel="DAPI (1)",
    shapes_layer="shapes_cells_instanseg_artifacts_filtered",
    table_layer="table_intensities_leiden",
    alpha=1,
    linewidth=0.6,
    cmap=cmap,
    column="ilastik_predicted_class_CD8a",
)

In [ ]:
# Plot probabilities
hp.pl.plot_shapes(
    sdata_crop,
    img_layer="REAscreen_IO_CRC_image",
    channel="DAPI (1)",
    shapes_layer="shapes_cells_instanseg_artifacts_filtered",
    table_layer="table_intensities_leiden",
    alpha=1,
    linewidth=0.6,
    cmap="coolwarm",
    column="ilastik_probability_pos_CD8a",
)

In [ ]:
# Plot probabilities
hp.pl.plot_shapes(
    sdata_crop,
    img_layer="REAscreen_IO_CRC_image",
    channel="DAPI (1)",
    shapes_layer="shapes_cells_instanseg_artifacts_filtered",
    table_layer="table_intensities_leiden",
    alpha=1,
    linewidth=0.6,
    cmap="coolwarm",
    column="ilastik_probability_neg_CD8a",
)

### 1.5 Assigning cell types

In [ ]:
annotation_table_path = os.path.join(output_path, "annotation_matrix.csv")

sdata_crop = assign_ilastik_cell_types(
    sdata_crop,
    annotation_table_path=annotation_table_path,
    table_layer="table_intensities_leiden",
    labels_layer="labels_cells_instanseg_artifacts_filtered",  # The label layer that corresponds to the data in the table layer.
    output_column="ilastik_cell_types",  # Name of output column
    default_value="other",  # Values used for all cells that do not fit any of the conditions.
)

In [ ]:
# Plot cell types
hp.pl.plot_shapes(
    sdata_crop,
    img_layer="REAscreen_IO_CRC_image",
    channel="DAPI (1)",
    shapes_layer="shapes_cells_instanseg_artifacts_filtered",
    table_layer="table_intensities_leiden",
    alpha=1,
    linewidth=0.6,
    cmap="rainbow",
    column="ilastik_cell_types",
)

## 2. Ilastik pixel classification
Creating pixel classifiers in Ilastik can be useful to semi-automatically annotate ROIs/artifacts/tissue.

### 2.1 Train pixel classifier
Open Ilastik (v.1.4.0) and create an new project: Pixel Classification.

Note that this documentation was written for version v.1.4.0, but everything seems to work with the latest version (v.1.4.1) as well.

**1. Input Data** </br> 
To load in a separate channel, you need to do the following:

- Under the tab `Raw Data` from `1. Input Data`, you would click `Add...` and select `Add separate Image(s)...`.
- Select the .h5 file containing the raw images and you will need to specify which channel you want to work on from the drop-down menu and click `OK` (this specifies the internal path in the h5 file).

To load in multiple channels, you need to specify a correct pattern that also includes the internal path (i.e. in the h5 file) to the correct images.

For example, to combine channel_1 and channel_2, you would do the following steps:
- Under the tab 'Raw Data' from '1. Input Data', you would click 'Add...' and select 'Add a single 3D/4D Volume from Sequence...'.
- Under `Specify Pattern`, you enter a patterns that specifies the images of interest. For example: `/path/to/images/raw_images.h5/channel_1; /path/to/images/raw_images.h5/channel_2` and click `Apply`. It is important that the path is specified correctly, multiple paths should be separated by a semicolon and the internal path in the h5 file should be specified as well in the path.
- Select `Stack Across: C` before clicking `OK`.

After adding the Raw Data to the ilastik project, it is useful to right-click on them, go to `Edit properties...` and make sure `Storage:` is set to `Copy into project file`. This makes sure, you can move around the ilastik project file (on your computer or even to other computers) without losing the link to the files the project was trained on (and risk losing your training). It is also useful to set `Nickname:` to something informative such as the name of the tissue/sample/replicate/etc (to keep track of which image is which). By default, this will be set to the filename.

<b>Important:</b> Note that the images for training in the Ilastik project can't be too large or Ilastik will crash. Around 10000 x 10000 pixels seems to work fine, but smaller (for training at least) is preferred. 

**2. Feature Selection** </br>
- Click `Select Features...`
- Select all features
- After clicking `OK`, wait until all features have been computed before moving on to the next step.

**3. Training** </br>
Here, you can create multiple classes for your classifier and train them until you are satisfied with the results. In general, it is useful to initially add a good amount of labels for the different classes for different regions of the image (or even already over multiple images) to capture the variation that is in the data and click `Live Update` to see the prediction results. Subsequently, you can focus more on the mistakes that are being made and add labels to correct those. When labeling, we suggest to unclick `Live Update` to avoid waiting time. It can be useful to use the `Uncertainty` layer to see which pixels are still not robustly trained for. When training, ilastik does not make it easy to change the visualization, but if you right-click on `Raw Input` and select `Adjust Thresholds`, you have some options to change the display.

Note that you can save your training labels by right-clicking `Labels` and subsequently `Export...`. Set the `File` to where you want to save the labels and under what name. This entire process needs to be done for every image separately, but it can be worth it to be able to recreate the ilastik project if something goes wrong or if we would want to preprocess the images for example.

**4. Prediction Export** </br>
- Under `Source`, we have a couple of options to export that can be useful, but we will select `Simple Segmentation` for now. 
- Click `Choose Export Image Settings...` and set Format to `tiff`. Change the File name to something informative and set `Convert to Data Type` to `integer 8-bit`.
- Click `OK` and click `Export All` to export the segmentation mask

**5. Batch processing** </br>
We will not use the batch processing tab since it is not possible to specify the internal paths in the h5 files.

### 2.2 Add results to SpatialData

In [ ]:
# Read in segmentation mask and clean up mask
from skimage import io, morphology

mask = io.imread(os.path.join(output_path, "ROI_segmentation.tiff"))
mask = mask == 1  # Keep only class 1 pixels
mask = morphology.remove_small_objects(mask, min_size=15000)
mask = morphology.remove_small_holes(mask, area_threshold=15000)
mask = mask.astype(np.uint8)  # Convert boolean mask back to integers

sdata_crop = hp.im.add_labels_layer(
    sdata_crop, arr=mask, output_layer="ilastik_mask", chunks=1024, overwrite=True
)

In [ ]:
# Plot mask
sdata_crop.pl.render_labels(
    "ilastik_mask",
).pl.show(coordinate_systems="global", figsize=(10, 10))